# Tes Stream NASA FIRMS (Karhutla Indonesia)

Notebook ini menguji FIRMS API sebagai sumber data stream ke layer **Bronze**:

1. Setup dan cek MAP_KEY
2. Cek ketersediaan data dan skema
3. Bandingkan sumber satelit (VIIRS SNPP / NOAA20 / NOAA21 / MODIS)
4. Stream (polling) + dedup + simpan ke Bronze
5. Analisis latensi near-real-time
6. Tes idempotensi dan error handling
7. Cek kualitas data dan peta sebaran

**Sebelum mulai:** simpan MAP_KEY di Colab Secrets (ikon kunci di sidebar kiri) dengan nama `FIRMS_MAP_KEY`, lalu aktifkan akses notebook.

## 1. Setup

In [ ]:
import os, io, json, time, glob
from pathlib import Path
import requests
import pandas as pd
import matplotlib.pyplot as plt

# ====== KONFIGURASI ======
USE_DRIVE = False          # True = simpan Bronze & state di Google Drive (awet saat runtime reset)
SOURCE = "VIIRS_SNPP_NRT"  # sumber utama untuk stream
BBOX = "95,-11,141,6"      # west,south,east,north (Indonesia)
DAY_RANGE = 1              # 1-5 hari ke belakang
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
# =========================

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/karhutla_lakehouse")
else:
    ROOT = Path("/content/karhutla_lakehouse")

BRONZE_DIR = ROOT / "data/bronze/firms"
STATE_FILE = ROOT / "data/state/firms_seen.json"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE.parent.mkdir(parents=True, exist_ok=True)

# Ambil MAP_KEY: Colab Secrets -> env -> input manual
try:
    from google.colab import userdata
    MAP_KEY = userdata.get("FIRMS_MAP_KEY")
except Exception:
    MAP_KEY = os.environ.get("FIRMS_MAP_KEY")
if not MAP_KEY:
    from getpass import getpass
    MAP_KEY = getpass("Masukkan FIRMS MAP_KEY: ")

print("Root:", ROOT)
print("MAP_KEY terbaca:", MAP_KEY[:4] + "..." + MAP_KEY[-4:])

## 2. Cek MAP_KEY dan ketersediaan data

Cek sisa kuota transaksi (batas sekitar 5.000 per 10 menit) dan rentang tanggal tiap sumber.

In [ ]:
r = requests.get(f"https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY={MAP_KEY}", timeout=30)
print("Status MAP_KEY:", r.status_code)
print(r.text[:500])

In [ ]:
r = requests.get(f"https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{MAP_KEY}/ALL", timeout=30)
print("Status:", r.status_code)
avail = pd.read_csv(io.StringIO(r.text))
avail[avail["data_id"].str.contains("NRT|SP", regex=True)]

## 3. Fungsi fetch dan tes satu kali tarik

FIRMS kadang membalas HTTP 200 dengan pesan error berupa teks biasa, jadi isi respons juga dicek.

In [ ]:
def fetch(source=SOURCE, bbox=BBOX, day_range=DAY_RANGE, timeout=60):
    url = f"{BASE_URL}/{MAP_KEY}/{source}/{bbox}/{day_range}"
    t0 = time.time()
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()
    if "latitude" not in resp.text[:300]:
        raise ValueError(f"Respons bukan CSV valid: {resp.text[:200]!r}")
    df = pd.read_csv(io.StringIO(resp.text))
    df.attrs["fetch_seconds"] = round(time.time() - t0, 2)
    return df

df = fetch()
print(f"Baris: {len(df)} | waktu fetch: {df.attrs['fetch_seconds']} detik")
df.head()

In [ ]:
print("== Skema ==")
print(df.dtypes)
print("\n== Null per kolom ==")
print(df.isna().sum())
print("\n== Duplikat penuh ==", df.duplicated().sum())
print("\n== Satelit ==")
print(df["satellite"].value_counts() if "satellite" in df else "-")
print("\n== Confidence ==")
print(df["confidence"].value_counts())
print("\n== Day/Night ==")
print(df["daynight"].value_counts())

## 4. Bandingkan sumber satelit

Untuk tiap sumber: jumlah titik, latensi, dan waktu respons API.

In [ ]:
def add_latency(d, now=None):
    now = now or pd.Timestamp.now(tz="UTC")
    t = d["acq_time"].astype(int).astype(str).str.zfill(4)
    acq = pd.to_datetime(d["acq_date"].astype(str) + " " + t, format="%Y-%m-%d %H%M", utc=True)
    d = d.copy()
    d["acq_ts_utc"] = acq
    d["latency_min"] = ((now - acq).dt.total_seconds() / 60).round(1)
    return d

SOURCES = ["VIIRS_SNPP_NRT", "VIIRS_NOAA20_NRT", "VIIRS_NOAA21_NRT", "MODIS_NRT"]
rows = []
for s in SOURCES:
    try:
        d = add_latency(fetch(source=s))
        rows.append({
            "source": s, "n_titik": len(d), "fetch_detik": d.attrs["fetch_seconds"],
            "latensi_min_menit": d["latency_min"].min() if len(d) else None,
            "latensi_median_menit": d["latency_min"].median() if len(d) else None,
            "latensi_max_menit": d["latency_min"].max() if len(d) else None,
        })
    except Exception as e:
        rows.append({"source": s, "n_titik": None, "error": str(e)[:80]})
    time.sleep(1)
pd.DataFrame(rows)

## 5. Stream: polling + dedup + simpan ke Bronze

Dedup key = `lat | lon | acq_date | acq_time | satelit`. State disimpan ke file, jadi tetap ada kalau notebook di-restart (selama folder tidak hilang).

In [ ]:
def make_key(row):
    return f"{row['latitude']}|{row['longitude']}|{row['acq_date']}|{int(row['acq_time']):04d}|{row.get('satellite', '')}"

def load_seen():
    return set(json.loads(STATE_FILE.read_text())) if STATE_FILE.exists() else set()

def save_seen(seen):
    STATE_FILE.write_text(json.dumps(sorted(seen)))

def write_bronze(d, source, now):
    part = BRONZE_DIR / f"source={source}" / f"ingest_date={now.strftime('%Y-%m-%d')}"
    part.mkdir(parents=True, exist_ok=True)
    out = part / f"batch_{now.strftime('%H%M%S')}.jsonl"
    d.to_json(out, orient="records", lines=True, date_format="iso")
    return out

def poll_once(seen, source=SOURCE):
    now = pd.Timestamp.now(tz="UTC")
    d = fetch(source=source)
    total = len(d)
    if total == 0:
        print(f"[{now:%H:%M:%S}] 0 baris dari API")
        return {"ts": now, "api": 0, "new": 0}
    d["_key"] = d.apply(make_key, axis=1)
    new = d[~d["_key"].isin(seen)].copy()
    stat = {"ts": now, "api": total, "new": len(new), "fetch_s": d.attrs["fetch_seconds"]}
    if len(new):
        new = add_latency(new, now)
        new["acq_ts_utc"] = new["acq_ts_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        new["ingested_at"] = now.strftime("%Y-%m-%dT%H:%M:%SZ")
        new["source"] = source
        path = write_bronze(new.drop(columns=["_key"]), source, now)
        seen.update(new["_key"])
        save_seen(seen)
        stat.update(lat_min=new["latency_min"].min(), lat_med=new["latency_min"].median(), lat_max=new["latency_min"].max())
        print(f"[{now:%H:%M:%S}] API={total} baru={len(new)} latensi median={stat['lat_med']} mnt -> {path.name}")
    else:
        print(f"[{now:%H:%M:%S}] API={total} baru=0")
    return stat

seen = load_seen()
print("Key sudah dikenal:", len(seen))

### Jalankan stream

Atur `N_ITER` dan `INTERVAL_S`. Untuk tes cepat pakai `N_ITER=3`, `INTERVAL_S=60`. Untuk melihat data baru benar-benar masuk, pakai interval 300 sampai 600 detik dan jumlah iterasi lebih banyak (sel akan berjalan selama itu).

In [ ]:
N_ITER = 3
INTERVAL_S = 60

history = []
backoff = 30
i = 0
while i < N_ITER:
    try:
        history.append(poll_once(seen))
        backoff = 30
        i += 1
        if i < N_ITER:
            time.sleep(INTERVAL_S)
    except Exception as e:
        print(f"[ERROR] {e} -> retry {backoff}s")
        time.sleep(backoff)
        backoff = min(backoff * 2, 600)

hist_df = pd.DataFrame(history)
hist_df

## 6. Analisis latensi dari data Bronze

In [ ]:
files = glob.glob(str(BRONZE_DIR / "**/*.jsonl"), recursive=True)
print("File Bronze:", len(files))
bronze = pd.concat((pd.read_json(f, lines=True) for f in files), ignore_index=True) if files else pd.DataFrame()
print("Total baris Bronze:", len(bronze))

if len(bronze):
    print(bronze["latency_min"].describe())
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    bronze["latency_min"].plot.hist(bins=40, ax=ax[0], title="Distribusi latensi (menit)")
    ax[0].set_xlabel("menit sejak akuisisi satelit")
    (bronze["latency_min"] / 60).plot.box(ax=ax[1], title="Latensi (jam)")
    plt.tight_layout(); plt.show()

## 7. Tes idempotensi

Tarik data yang sama dua kali berturut-turut. Hasil yang diharapkan: titik baru = **0** (tidak ada duplikat).

In [ ]:
before = len(seen)
s1 = poll_once(seen)
s2 = poll_once(seen)
print(f"Setelah polling ke-2 berturut-turut -> baru = {s2['new']}")
assert s2["new"] == 0, "Dedup GAGAL: masih ada duplikat"
print("LOLOS tes idempotensi")

if len(bronze):
    keys = bronze.apply(make_key, axis=1)
    print("Duplikat di Bronze:", keys.duplicated().sum())

## 8. Tes error handling

Simulasi MAP_KEY salah dan timeout. Pastikan kedua kasus ditangkap sebagai error (bukan data kosong yang diam-diam lolos).

In [ ]:
# 1) MAP_KEY salah
real_key = MAP_KEY
MAP_KEY = "KEY_SALAH_123"
try:
    fetch()
    print("TIDAK ADA ERROR -> perlu dicek!")
except Exception as e:
    print("OK, error tertangkap (key salah):", str(e)[:120])
MAP_KEY = real_key

# 2) Timeout
try:
    fetch(timeout=0.001)
    print("TIDAK ADA ERROR -> perlu dicek!")
except Exception as e:
    print("OK, error tertangkap (timeout):", type(e).__name__)

## 9. Cek kualitas data Bronze

In [ ]:
if len(bronze):
    q = bronze.copy()
    # Konversi waktu akuisisi ke WIB
    q["acq_ts_utc"] = pd.to_datetime(q["acq_ts_utc"], utc=True)
    q["acq_ts_wib"] = q["acq_ts_utc"].dt.tz_convert("Asia/Jakarta")

    checks = {
        "lat dalam rentang Indonesia (-11..6)": q["latitude"].between(-11, 6).mean(),
        "lon dalam rentang Indonesia (95..141)": q["longitude"].between(95, 141).mean(),
        "frp tidak null": q["frp"].notna().mean(),
        "frp >= 0": (q["frp"] >= 0).mean(),
        "confidence tidak null": q["confidence"].notna().mean(),
        "latensi >= 0": (q["latency_min"] >= 0).mean(),
    }
    print(pd.Series(checks).map(lambda v: f"{v*100:.1f}%"))
    print("\nRentang waktu (WIB):", q["acq_ts_wib"].min(), "->", q["acq_ts_wib"].max())
    print("\nDistribusi confidence:\n", q["confidence"].value_counts())
    print("\nFRP:\n", q["frp"].describe())

## 10. Peta sebaran hotspot

In [ ]:
if len(bronze):
    fig, ax = plt.subplots(figsize=(11, 5))
    sc = ax.scatter(bronze["longitude"], bronze["latitude"], c=bronze["frp"], s=8, cmap="YlOrRd", vmax=bronze["frp"].quantile(0.95))
    ax.set_xlim(94, 142); ax.set_ylim(-12, 7)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title(f"Hotspot FIRMS ({SOURCE}) - warna = FRP")
    plt.colorbar(sc, label="FRP (MW)"); plt.show()

## 11. Ringkasan checklist

Isi sendiri setelah menjalankan notebook:

| Aspek | Hasil | Catatan |
|---|---|---|
| MAP_KEY valid dan kuota aman | | |
| Sumber data terbaik (jumlah titik, latensi) | | |
| Latensi median (menit) | | |
| Skema dan null | | |
| Dedup / idempotensi | | |
| Error handling (key salah, timeout) | | |
| Kualitas data (rentang geo, FRP, confidence) | | |
| Siap diteruskan ke Silver | | |